<a href="https://colab.research.google.com/github/frank-morales2020/MLxDL/blob/main/Claude_4_6_%2B_JEPA_%2B_H2E_Integration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install anthropic -q
!pip install colab-env -q

In [ ]:
import colab_env
colab_env.RELOAD()

## CASE1

In [ ]:
import os
import logging
import warnings
import torch
import torch.nn as nn
import anthropic
import numpy as np

# ─── 0. SILENCE LOGGING & GLOBAL SEEDING ──────────────────────────────────────
logging.getLogger("anthropic").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

SEED = 123
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─── 1. SETUP CLAUDE OPUS 4.6 ────────────────────────────────────────────────
MODEL_NAME = "claude-opus-4-6"
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is not set.")

client = anthropic.Anthropic(api_key=api_key)

# ─── 2. JEPA WORLD MODEL (LATENT DYNAMICS) ───────────────────────────────────
class HumanoidJEPA(nn.Module):
    def __init__(self, in_dim=22, latent_dim=512):
        super().__init__()
        # Apply SEED to the specific layer initialization for reproducibility
        torch.manual_seed(SEED)

        # Context Encoder: 22-DoF telemetry -> abstract embeddings
        self.encoder = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, latent_dim)
        )
        # Predictor: The "World Model" predicting physics in latent space
        self.predictor = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )

    def forward(self, x):
        z_t = self.encoder(x)
        z_t1_pred = self.predictor(z_t)
        return z_t, z_t1_pred

# ─── 3. H2E REASONING CYCLE ───────────────────────────────────────────────────
def execute_h2e_jepa_cycle(telemetry, goal):
    # Initialize the JEPA World Model
    jepa = HumanoidJEPA()

    # 1. Prediction: Generate latent representations (Deterministic via SEED)
    state_tensor = torch.FloatTensor(telemetry)
    z_curr, z_pred = jepa(state_tensor)

    # Convert embeddings to sampled strings for the prompt
    curr_sample = z_curr.detach().numpy()[:5].tolist()
    pred_sample = z_pred.detach().numpy()[:5].tolist()

    # 2. Reasoning: Invoke Claude Opus 4.6
    # CORRECTED 2026 SCHEMA: effort is now in output_config
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=8192,
        thinking={
            "type": "adaptive"
        },
        output_config={
            "effort": "max"  # Highest reasoning tier for 22-DoF stability
        },
        messages=[{
            "role": "user",
            "content": f"""
            [H2E ACCOUNTABILITY PROTOCOL]
            System: 22-DoF Humanoid. Global Seed: {SEED}
            Goal: {goal}

            Current Latent (z_t): {curr_sample}...
            Predicted Latent (z_t+1): {pred_sample}...

            Analyze the latent transition for Center of Mass (CoM) stability.
            If valid, provide the 22-DoF joint vector for the next step.
            """
        }]
    )
    return response

# ─── 4. MAIN ──────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # Generate reproducible telemetry data
    current_telemetry = np.random.uniform(-1, 1, 22)

    print(f"--- Querying {MODEL_NAME} (Seed: {SEED}) ---")
    try:
        result = execute_h2e_jepa_cycle(current_telemetry, "Balanced forward step")

        for block in result.content:
            if block.type == "thinking":
                print(f"\n[CLAUDE OPUS 4.6 THINKING]:\n{block.thinking}")
            elif block.type == "text":
                print(f"\n[H2E VERIFIED ACTION]:\n{block.text}")
    except Exception as e:
        print(f"Error encountered: {e}")

--- Querying claude-opus-4-6 (Seed: 123) ---

[CLAUDE OPUS 4.6 THINKING]:
The user is asking me to analyze a latent transition for a 22-DoF humanoid robot's Center of Mass (CoM) stability as part of an "H2E Accountability Protocol." Let me break this down carefully.

First, let me analyze the latent transition:

**Current Latent (z_t):** [0.01595, -0.28614, -0.08616, 0.25751, 0.28637]...
**Predicted Latent (z_t+1):** [0.18872, -0.00964, 0.06177, -0.02938, 0.11246]...


Looking at the component-wise changes, I see the first latent dimension shifts significantly positive by about 0.17, the second dimension makes a large positive movement toward zero from -0.29, the third shows a moderate positive change of roughly 0.15, and the fourth dimension drops substantially by about 0.29 also moving toward zero.

The fifth dimension follows a similar pattern with a moderate negative shift of around 0.17 toward zero. What's striking across all these movements is that the latent space is contracting

## CASE2

In [ ]:
import os
import csv
import time
import logging
import warnings
import torch
import torch.nn as nn
import anthropic
import numpy as np
from datetime import datetime

# ─── 0. SILENCE LOGGING & GLOBAL SEEDING ──────────────────────────────────────
logging.getLogger("anthropic").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

SEED = 123
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ─── 1. SETUP CLAUDE OPUS 4.6 (MARCH 2026 SPECS) ─────────────────────────────
MODEL_NAME = "claude-opus-4-6"
api_key = os.environ.get("ANTHROPIC_API_KEY")
if not api_key:
    raise ValueError("ANTHROPIC_API_KEY is not set.")

client = anthropic.Anthropic(api_key=api_key)

# ─── 2. JEPA WORLD MODEL WITH EMA TARGET ENCODER ──────────────────────────────
class HumanoidJEPA(nn.Module):
    def __init__(self, in_dim=22, latent_dim=512, ema_decay=0.99):
        super().__init__()
        torch.manual_seed(SEED)
        self.ema_decay = ema_decay

        # Context Encoder (x -> z_t)
        self.context_encoder = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, latent_dim)
        )

        # Target Encoder (Momentum Teacher - No Gradients)
        self.target_encoder = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, latent_dim)
        )
        # Initialize target weights to match context
        self.target_encoder.load_state_dict(self.context_encoder.state_dict())
        for param in self.target_encoder.parameters():
            param.requires_grad = False

        # Predictor: The "World Model" (z_t -> predicted z_t1)
        self.predictor = nn.Sequential(
            nn.Linear(latent_dim, latent_dim),
            nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )

    @torch.no_grad()
    def update_target_encoder(self):
        """EMA update to keep the target representations stable."""
        for c_param, t_param in zip(self.context_encoder.parameters(), self.target_encoder.parameters()):
            t_param.data = self.ema_decay * t_param.data + (1 - self.ema_decay) * c_param.data

    def forward(self, x_curr, x_next=None):
        z_t = self.context_encoder(x_curr)
        z_t1_pred = self.predictor(z_t)

        z_t1_target = None
        if x_next is not None:
            z_t1_target = self.target_encoder(x_next)

        return z_t, z_t1_pred, z_t1_target

# ─── 3. H2E AUDIT & MISSION LOGGING ───────────────────────────────────────────
class MissionLogger:
    def __init__(self, filename="h2e_mission_audit.csv"):
        self.filename = filename
        if not os.path.exists(self.filename):
            with open(self.filename, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["Timestamp", "Goal", "SROI_Status", "Latent_Delta", "Command"])

    def log_action(self, goal, status, delta, command):
        with open(self.filename, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([datetime.now().isoformat(), goal, status, f"{delta:.4f}", command[:50] + "..."])

# ─── 4. H2E REASONING CYCLE ───────────────────────────────────────────────────
def execute_h2e_jepa_cycle(telemetry, goal, logger):
    jepa = HumanoidJEPA()

    # 1. Prediction (Deterministic via SEED 123)
    state_tensor = torch.FloatTensor(telemetry)
    z_curr, z_pred, _ = jepa(state_tensor)

    # Calculate Latent Delta for stability audit
    latent_delta = torch.norm(z_pred - z_curr).item()

    # 2. Reasoning: Invoke Claude Opus 4.6 (Adaptive Thinking)
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=8192,
        thinking={"type": "adaptive"},
        output_config={"effort": "max"}, # Max reasoning for 22-DoF stability
        messages=[{
            "role": "user",
            "content": f"""
            [H2E ACCOUNTABILITY PROTOCOL]
            22-DoF Humanoid (Seed: {SEED})
            Current Latent (z_t): {z_curr.detach().numpy()[:5].tolist()}
            Predicted Latent (z_t+1): {z_pred.detach().numpy()[:5].tolist()}
            Goal: {goal}

            Audit the transition. Output verified joint vectors for next step.
            """
        }]
    )

    # 3. Industrial Audit Logging
    status = "VERIFIED" if latent_delta < 0.6 else "DRIFT_DETECTED"
    logger.log_action(goal, status, latent_delta, response.content[1].text if len(response.content) > 1 else "N/A")

    return response

# ─── 5. MAIN EXECUTION ───────────────────────────────────────────────────────
if __name__ == "__main__":
    audit_logger = MissionLogger()
    mock_telemetry = np.random.uniform(-1, 1, 22)

    print(f"--- Querying {MODEL_NAME} with H2E Governance ---")

    try:
        result = execute_h2e_jepa_cycle(mock_telemetry, "Stair ascent (10cm step)", audit_logger)

        for block in result.content:
            if block.type == "thinking":
                print(f"\n[INTERNAL AUDIT]:\n{block.thinking}")
            elif block.type == "text":
                print(f"\n[H2E VERIFIED ACTION]:\n{block.text}")

        print(f"\nAudit saved to: {audit_logger.filename}")

    except Exception as e:
        print(f"Critical System Error: {e}")

--- Querying claude-opus-4-6 with H2E Governance ---

[INTERNAL AUDIT]:
The user is presenting what appears to be an "H2E Accountability Protocol" for a 22-DoF (Degrees of Freedom) humanoid robot simulation. They want me to audit a latent space transition and output verified joint vectors for a stair ascent task.

Let me analyze this carefully:

1. **System**: 22-DoF Humanoid with seed 123
2. **Current latent state** (z_t): 5-dimensional latent vector
3. **Predicted latent state** (z_t+1): 5-dimensional latent vector
4. **Task**: Stair ascent with 10cm step height


5. **Latent space deltas**: I'm computing the component-wise changes across the five dimensions, with the largest shift occurring in dimension 1 at roughly 0.32 units, while dimension 4 shows a smaller change of about 0.036 units.

Now I'm thinking through the degrees of freedom for a 22-DoF humanoid during stair climbing. The configuration breaks down into leg articulation with hip, knee, and ankle joints providing the pri

In [ ]:
from google.colab import files
files.download('/content/h2e_mission_audit.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Comprehensive JEPA with EMA & H2E Governance

https://www.youtube.com/watch?v=g0LBEjZftkw

https://www.youtube.com/watch?v=Y_9tAf3eN9s&t=1s

In [ ]:
# Install MuJoCo and its viewer for Colab
!pip install mujoco -q
!pip install mujoco-python-viewer -q

# Set up the software rendering for Colab's headless environment
import os
os.environ['MUJOCO_GL'] = 'egl'

In [ ]:
# Install dependencies for AI, Environment, and Physics
!pip install anthropic mujoco mediapy colab-env -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 19.7 MB/s eta 0:00:00


In [ ]:
import os
import random
import csv
import logging
import warnings
import torch
import torch.nn as nn
import anthropic
import numpy as np
import mujoco
from datetime import datetime

# --- 1. SMLA REPRODUCIBILITY & SILENT AUDIT ---
def set_reproducibility(seed=123):
    """Ensures deterministic behavior across Python, NumPy, and PyTorch."""
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Execute reproducibility for SMLA compliance
set_reproducibility(123)

# Silence non-critical logs for a clean H2E audit trail
logging.getLogger("anthropic").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
os.environ['MUJOCO_GL'] = 'egl' # Hardware-accelerated headless rendering

# --- 2. JEPA WORLD MODEL (LATENT DYNAMICS) ---
class HumanoidJEPA(nn.Module):
    def __init__(self, in_dim=22, latent_dim=512, ema_decay=0.99):
        super().__init__()
        self.ema_decay = ema_decay

        # Context Encoder (x_t -> z_t)
        self.context_encoder = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, latent_dim)
        )

        # Target Encoder (Momentum Teacher)
        self.target_encoder = nn.Sequential(
            nn.Linear(in_dim, 256), nn.LayerNorm(256), nn.GELU(),
            nn.Linear(256, latent_dim)
        )
        self.target_encoder.load_state_dict(self.context_encoder.state_dict())
        for p in self.target_encoder.parameters(): p.requires_grad = False

        # Predictor (z_t -> predicted z_t1)
        self.predictor = nn.Sequential(
            nn.Linear(latent_dim, latent_dim), nn.ReLU(),
            nn.Linear(latent_dim, latent_dim)
        )

    def forward(self, x_curr):
        z_t = self.context_encoder(x_curr)
        z_pred_t1 = self.predictor(z_t)
        return z_t, z_pred_t1

# --- 3. H2E MISSION LOGGING ---
class MissionAuditor:
    def __init__(self, filename="h2e_mission_audit.csv"):
        self.filename = filename
        if not os.path.exists(self.filename):
            with open(self.filename, mode='w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow(["Timestamp", "Goal", "Status", "Latent_Delta", "Verified_Joints"])

    def log(self, goal, status, delta, joints):
        with open(self.filename, mode='a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([datetime.now().isoformat(), goal, status, f"{delta:.4f}", joints[:60] + "..."])

# --- 4. H2E REASONING & EXECUTION ---
def execute_h2e_jepa_cycle(telemetry, goal):
    jepa = HumanoidJEPA()
    auditor = MissionAuditor()
    client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))

    # 1. JEPA Prediction (Deterministic via SMLA set_reproducibility)
    state_tensor = torch.FloatTensor(telemetry)
    z_curr, z_pred = jepa(state_tensor)
    l_delta = torch.norm(z_pred - z_curr).item()

    # 2. Claude Opus 4.6 Expert Reasoning
    response = client.messages.create(
        model="claude-opus-4-6",
        max_tokens=8192,
        thinking={"type": "adaptive"},
        output_config={"effort": "max"}, # High-fidelity physics reasoning
        messages=[{
            "role": "user",
            "content": f"[H2E AUDIT] Goal: {goal}. Latent Delta: {l_delta:.4f}. Audit 22-DoF stability & 120Nm torque limits."
        }]
    )

    # 3. Industrial Audit & Status
    status = "VERIFIED" if l_delta < 8.7 else "REPLAN_REQUIRED"
    auditor.log(goal, status, l_delta, response.content[1].text if len(response.content) > 1 else "N/A")

    return response

# --- 5. EXECUTION ---
if __name__ == "__main__":
    # Test deterministic initialization
    test_tensor = torch.randn(2, 2)
    print(f"SMLA Verification Tensor (Seed 123):\n{test_tensor}\n")

    # Run humanoid control loop
    mock_telemetry = np.random.uniform(-1, 1, 22)
    result = execute_h2e_jepa_cycle(mock_telemetry, "Stair ascent (10cm)")

    for block in result.content:
        if block.type == "text":
            print(f"\n[H2E VERIFIED COMMANDS]:\n{block.text}")

SMLA Verification Tensor (Seed 123):
tensor([[-0.1115,  0.1204],
        [-0.3696, -0.2404]])


[H2E VERIFIED COMMANDS]:
# H2E Stair Ascent Audit Report

## Task Parameters
| Parameter | Value |
|---|---|
| **Maneuver** | Stair ascent |
| **Step height** | 10 cm |
| **DoF model** | 22 |
| **Torque hard-limit** | 120 Nm (all joints) |
| **Latent delta (Δz)** | 8.7730 |

---

## 1. 22-DoF Joint Map & Grouping

```
LOWER BODY (12 DoF)            UPPER BODY (10 DoF)
├─ L_Hip   [roll/pitch/yaw]    ├─ Waist  [roll/pitch/yaw]  ← 3 DoF
├─ L_Knee  [pitch]             ├─ L_Arm  [sh_pitch/sh_roll/elbow]
├─ L_Ankle [roll/pitch]        └─ R_Arm  [sh_pitch/sh_roll/elbow/wrist]
├─ R_Hip   [roll/pitch/yaw]         
├─ R_Knee  [pitch]                  Total: 12 + 10 = 22 DoF
└─ R_Ankle [roll/pitch]
```

---

## 2. Latent Delta Analysis — ⚠️ FLAG

**Δz = 8.7730** is evaluated against typical policy operating ranges:

| Regime | Δz Range | Status |
|---|---|---|
| Flat walk (nominal) | 0.5 – 2.5 | Baseli

## Final Sovereign AI Stack: SMLA + JEPA + H2E

In [ ]:
"""
H2E (Human-to-Exoskeleton) Governance System
Industrial-grade safety framework for humanoid robot control
Version: 2.0.2 - REPRODUCIBILITY FIXED
"""

import os
import sys
import json
import random
import csv
import logging
import warnings
import re
import time
from datetime import datetime
from dataclasses import dataclass, asdict
from typing import Optional, Dict, List, Tuple, Any
from enum import Enum

import torch
import torch.nn as nn
import numpy as np
import anthropic

# ============================================================================
# CONFIGURATION & CONSTANTS
# ============================================================================

@dataclass(frozen=True)
class H2EConfig:
    """Immutable system configuration"""
    # Reproducibility
    SEED: int = 123

    # Model dimensions
    SENSOR_DIM: int = 22
    LATENT_DIM: int = 512

    # Safety thresholds
    MAX_LATENT_DELTA: float = 4.0
    MAX_TORQUE_NM: float = 120.0
    TORQUE_WARNING_THRESHOLD: float = 0.9  # 90% of limit

    # Biomechanical models (Nm/kg)
    TORQUE_PER_KG = {
        'ascent': 1.75,    # 1.5-2.0 range
        'descent': 2.0,     # Higher eccentric load
        'level': 0.8,       # Level walking
        'unknown': 1.75
    }

    # Reference step height (meters)
    REF_STEP_HEIGHT: float = 0.20

    # API settings
    CLAUDE_MODELS = [
        "claude-3-opus-20240229",
        "claude-3-sonnet-20240229",
        "claude-3-haiku-20240307"
    ]
    MAX_RETRIES: int = 3
    RETRY_DELAY: float = 1.0

    # Logging
    AUDIT_FILE: str = "h2e_mission_audit.csv"
    METRICS_FILE: str = "h2e_metrics.json"


# ============================================================================
# REPRODUCIBILITY ENGINE
# ============================================================================

class ReproducibilityEngine:
    """Ensures 100% deterministic behavior across all components"""

    @staticmethod
    def set_seed(seed: int = 123):
        """Lock all random number generators to deterministic state"""
        random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
        torch.use_deterministic_algorithms(True)

    @staticmethod
    def verify(seed: int = 123) -> bool:
        """Verify reproducibility with a clean seed state"""
        # Set seed fresh
        ReproducibilityEngine.set_seed(seed)

        # Generate verification tensor
        test_tensor = torch.randn(2, 2)
        expected = torch.tensor([[-0.1115, 0.1204], [-0.3696, -0.2404]])

        matches = torch.allclose(test_tensor, expected, rtol=1e-4, atol=1e-4)

        print(f"\n🔍 SMLA Verification:")
        print(f"   Generated: {test_tensor.tolist()}")
        print(f"   Expected:  {expected.tolist()}")
        print(f"   Status:    {'✅ VERIFIED' if matches else '❌ MISMATCH'}")

        return matches


# ============================================================================
# JEPA WORLD MODEL - WITH EXPLICIT SEED MANAGEMENT
# ============================================================================

class HumanoidJEPA(nn.Module):
    """
    Joint Embedding Predictive Architecture for humanoid dynamics
    Uses EMA teacher for stable representation learning
    """

    def __init__(self, config: H2EConfig):
        super().__init__()
        # Save current RNG state
        rng_state = torch.get_rng_state()
        random_state = random.getstate()
        np_state = np.random.get_state()

        # Set seed for this initialization only
        ReproducibilityEngine.set_seed(config.SEED)

        # Encoder networks
        self.context_encoder = nn.Sequential(
            nn.Linear(config.SENSOR_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, config.LATENT_DIM)
        )

        self.target_encoder = nn.Sequential(
            nn.Linear(config.SENSOR_DIM, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Linear(256, config.LATENT_DIM)
        )

        # Initialize target encoder with context weights
        self.target_encoder.load_state_dict(self.context_encoder.state_dict())
        for p in self.target_encoder.parameters():
            p.requires_grad = False

        # Predictor network
        self.predictor = nn.Sequential(
            nn.Linear(config.LATENT_DIM, config.LATENT_DIM),
            nn.ReLU(),
            nn.Linear(config.LATENT_DIM, config.LATENT_DIM)
        )

        # Restore original RNG state
        torch.set_rng_state(rng_state)
        random.setstate(random_state)
        np.random.set_state(np_state)

    def forward(self, x_curr: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Encode current state and predict next state in latent space"""
        z_curr = self.context_encoder(x_curr)
        z_pred = self.predictor(z_curr)
        return z_curr, z_pred

    @torch.no_grad()
    def compute_delta(self, telemetry: np.ndarray) -> float:
        """Compute latent space delta for safety checking"""
        x = torch.FloatTensor(telemetry)
        z_curr, z_pred = self.forward(x)
        return torch.norm(z_pred - z_curr).item()


# ============================================================================
# PHYSICS ENGINE
# ============================================================================

class TorqueEstimator:
    """Physics-based torque estimation using biomechanical models"""

    def __init__(self, config: H2EConfig):
        self.config = config

    def estimate_knee_torque(self, user_mass_kg: float, step_height_m: float,
                            gait_type: str) -> float:
        """
        Estimate peak knee torque based on biomechanical literature

        Args:
            user_mass_kg: User mass in kg
            step_height_m: Step height in meters
            gait_type: 'ascent', 'descent', or 'level'

        Returns:
            Estimated peak torque in Nm
        """
        # Get base torque per kg for this gait type
        torque_per_kg = self.config.TORQUE_PER_KG.get(
            gait_type.lower(),
            self.config.TORQUE_PER_KG['unknown']
        )

        # Scale factor for step height (normalized to reference)
        height_factor = min(step_height_m / self.config.REF_STEP_HEIGHT, 1.5)

        # Calculate estimated torque
        estimated_torque = user_mass_kg * torque_per_kg * height_factor

        return estimated_torque

    def calculate_safety_margin(self, estimated_torque: float) -> float:
        """Calculate safety margin relative to max torque"""
        return self.config.MAX_TORQUE_NM - estimated_torque

    def get_safety_status(self, estimated_torque: float) -> Dict[str, Any]:
        """Determine safety status with severity levels"""
        margin = self.calculate_safety_margin(estimated_torque)

        if estimated_torque > self.config.MAX_TORQUE_NM:
            return {
                'safe': False,
                'severity': 'CRITICAL',
                'message': f'Torque exceeds limit by {-margin:.1f} Nm',
                'action': 'IMMEDIATE_STOP'
            }
        elif estimated_torque > self.config.MAX_TORQUE_NM * self.config.TORQUE_WARNING_THRESHOLD:
            return {
                'safe': True,
                'severity': 'WARNING',
                'message': f'Approaching torque limit ({estimated_torque:.1f}/{self.config.MAX_TORQUE_NM} Nm)',
                'action': 'MONITOR_CLOSELY'
            }
        else:
            return {
                'safe': True,
                'severity': 'NOMINAL',
                'message': f'Within safety margins ({margin:.1f} Nm buffer)',
                'action': 'PROCEED'
            }


# ============================================================================
# SCENARIO PARSER
# ============================================================================

@dataclass
class Scenario:
    """Parsed scenario information"""
    description: str
    gait_type: str
    step_height_m: float
    user_mass_kg: int
    has_heavy_load: bool

    @classmethod
    def from_goal(cls, goal: str, default_mass: int = 80) -> 'Scenario':
        """Parse goal string into structured scenario"""
        goal_lower = goal.lower()

        # Determine gait type
        if 'descent' in goal_lower:
            gait_type = 'descent'
        elif 'level' in goal_lower:
            gait_type = 'level'
        else:
            gait_type = 'ascent'

        # Extract step height
        step_height_m = 0.10  # default
        height_match = re.search(r'(\d+)cm', goal)
        if height_match:
            step_height_m = int(height_match.group(1)) / 100.0

        # Check for heavy load
        has_heavy_load = 'heavy load' in goal_lower

        # Adjust mass for heavy load
        user_mass_kg = 100 if has_heavy_load else default_mass

        return cls(
            description=goal,
            gait_type=gait_type,
            step_height_m=step_height_m,
            user_mass_kg=user_mass_kg,
            has_heavy_load=has_heavy_load
        )


# ============================================================================
# SAFETY GOVERNOR
# ============================================================================

class SafetyGovernor:
    """Multi-layer safety system with staged transitions"""

    def __init__(self, config: H2EConfig):
        self.config = config

    def apply_staged_transition(self, raw_delta: float, torque_safe: bool) -> Tuple[float, bool]:
        """
        Apply staged transition protocol with safety constraints

        Returns:
            Tuple of (clamped_delta, veto_triggered)
        """
        veto_triggered = False
        clamped_delta = raw_delta

        # Stage 1: Latent space clamping
        if raw_delta > self.config.MAX_LATENT_DELTA:
            clamped_delta = self.config.MAX_LATENT_DELTA
            veto_triggered = True

        # Stage 2: Torque-based reduction
        if not torque_safe:
            clamped_delta = min(clamped_delta, self.config.MAX_LATENT_DELTA * 0.5)
            veto_triggered = True

        return clamped_delta, veto_triggered


# ============================================================================
# AUDIT SYSTEM
# ============================================================================

class AuditSystem:
    """Comprehensive audit logging for regulatory compliance"""

    def __init__(self, config: H2EConfig):
        self.config = config
        self.session_id = datetime.now().strftime('%Y%m%d_%H%M%S')
        self._init_csv()
        self.metrics = []

    def _init_csv(self):
        """Initialize CSV file with headers"""
        if not os.path.exists(self.config.AUDIT_FILE):
            with open(self.config.AUDIT_FILE, 'w', newline='') as f:
                writer = csv.writer(f)
                writer.writerow([
                    'Timestamp', 'SessionID', 'Goal', 'GaitType',
                    'StepHeight_cm', 'UserMass_kg', 'LatentDelta',
                    'RawDelta', 'VetoTriggered', 'EstimatedTorque_Nm',
                    'TorqueLimit_Nm', 'SafetyMargin_Nm', 'SafetyStatus',
                    'ClaudeAvailable'
                ])

    def log_decision(self, scenario: Scenario, raw_delta: float, clamped_delta: float,
                    veto: bool, torque: float, safety_status: Dict, claude_used: bool):
        """Log a safety decision to CSV"""
        with open(self.config.AUDIT_FILE, 'a', newline='') as f:
            writer = csv.writer(f)
            writer.writerow([
                datetime.now().isoformat(),
                self.session_id,
                scenario.description,
                scenario.gait_type,
                f"{scenario.step_height_m*100:.0f}",
                scenario.user_mass_kg,
                f"{clamped_delta:.3f}",
                f"{raw_delta:.3f}",
                veto,
                f"{torque:.1f}",
                self.config.MAX_TORQUE_NM,
                f"{self.config.MAX_TORQUE_NM - torque:.1f}",
                safety_status['severity'],
                claude_used
            ])

    def record_metrics(self, metrics: Dict):
        """Store metrics for later analysis"""
        self.metrics.append({
            'timestamp': datetime.now().isoformat(),
            **metrics
        })

    def save_metrics(self):
        """Save metrics to JSON file"""
        with open(self.config.METRICS_FILE, 'w') as f:
            json.dump({
                'session_id': self.session_id,
                'metrics': self.metrics
            }, f, indent=2)


# ============================================================================
# DIAGNOSTIC ENGINE
# ============================================================================

class DiagnosticEngine:
    """Generates diagnostic reports with or without Claude API"""

    def __init__(self, config: H2EConfig):
        self.config = config
        self.client = None
        self._init_claude()

    def _init_claude(self):
        """Initialize Claude client if API key available"""
        api_key = os.environ.get("ANTHROPIC_API_KEY")
        if api_key:
            try:
                self.client = anthropic.Anthropic(api_key=api_key)
                print("✅ Claude API initialized")
            except Exception as e:
                print(f"⚠ Claude API init failed: {e}")

    def generate_report(self, scenario: Scenario, raw_delta: float, clamped_delta: float,
                       veto: bool, torque: float, safety_status: Dict) -> str:
        """Generate diagnostic report using Claude or simulation"""

        if self.client:
            return self._generate_claude_report(scenario, raw_delta, clamped_delta,
                                                veto, torque, safety_status)
        else:
            return self._generate_simulated_report(scenario, raw_delta, clamped_delta,
                                                   veto, torque, safety_status)

    def _generate_claude_report(self, scenario: Scenario, raw_delta: float,
                                clamped_delta: float, veto: bool, torque: float,
                                safety_status: Dict) -> str:
        """Generate report using Claude API with fallback"""

        prompt = f"""[H2E SAFETY AUDIT]
Mission: {scenario.description}
Gait: {scenario.gait_type}
Step Height: {scenario.step_height_m*100:.0f}cm
User Mass: {scenario.user_mass_kg}kg

CONTROL METRICS:
- Raw Latent Delta: {raw_delta:.2f}
- Clamped Delta: {clamped_delta:.2f} (max safe: {self.config.MAX_LATENT_DELTA})
- Veto Active: {veto}

PHYSICS ESTIMATES:
- Estimated Peak Torque: {torque:.1f} Nm
- Torque Limit: {self.config.MAX_TORQUE_NM} Nm
- Safety Margin: {self.config.MAX_TORQUE_NM - torque:.1f} Nm
- Status: {safety_status['severity']}

Biomechanical Reference (literature):
- Stair ascent: 1.5-2.0 Nm/kg → {scenario.user_mass_kg*1.5:.0f}-{scenario.user_mass_kg*2.0:.0f} Nm
- Step height demand: {'LOW' if scenario.step_height_m<=0.15 else 'MODERATE' if scenario.step_height_m<=0.20 else 'HIGH'}

Provide concise safety assessment and specific actionable recommendations."""

        # Try each Claude model with retries
        for model in self.config.CLAUDE_MODELS:
            for attempt in range(self.config.MAX_RETRIES):
                try:
                    response = self.client.messages.create(
                        model=model,
                        max_tokens=1024,
                        messages=[{"role": "user", "content": prompt}]
                    )
                    return response.content[0].text
                except Exception as e:
                    if attempt == self.config.MAX_RETRIES - 1:
                        continue
                    time.sleep(self.config.RETRY_DELAY * (attempt + 1))

        return self._generate_simulated_report(scenario, raw_delta, clamped_delta,
                                               veto, torque, safety_status)

    def _generate_simulated_report(self, scenario: Scenario, raw_delta: float,
                                   clamped_delta: float, veto: bool, torque: float,
                                   safety_status: Dict) -> str:
        """Generate simulated report when Claude unavailable"""

        # Safety icon and header
        icons = {
            'CRITICAL': '❌ CRITICAL',
            'WARNING': '⚠ WARNING',
            'NOMINAL': '✓ NOMINAL'
        }

        torque_range = f"{scenario.user_mass_kg*1.5:.0f}-{scenario.user_mass_kg*2.0:.0f}"

        report = f"""
╔════════════════════════════════════════════════════════════╗
║     H2E SAFETY ANALYSIS REPORT (SIMULATED MODE)           ║
╚════════════════════════════════════════════════════════════╝

MISSION: {scenario.description}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ Gait:         {scenario.gait_type.upper()}
│ Step Height:  {scenario.step_height_m*100:.0f}cm ({'LOW' if scenario.step_height_m<=0.15 else 'MODERATE' if scenario.step_height_m<=0.20 else 'HIGH'} demand)
│ User Mass:    {scenario.user_mass_kg}kg {'(with heavy load)' if scenario.has_heavy_load else ''}

SAFETY STATUS: {icons[safety_status['severity']]}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ {safety_status['message']}
│ Recommended Action: {safety_status['action']}

PHYSICS ESTIMATES
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ Estimated Torque:  {torque:.1f} Nm
│ Torque Limit:      {self.config.MAX_TORQUE_NM} Nm
│ Safety Margin:     {self.config.MAX_TORQUE_NM - torque:.1f} Nm
│ Literature Range:  {torque_range} Nm (for {scenario.user_mass_kg}kg)

CONTROL METRICS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
│ Raw Delta:     {raw_delta:.2f}
│ Clamped Delta: {clamped_delta:.2f} (max: {self.config.MAX_LATENT_DELTA})
│ Veto System:   {'ACTIVE' if veto else 'INACTIVE'}

RECOMMENDATIONS
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
"""

        if safety_status['severity'] == 'CRITICAL':
            report += """│ 1. ❌ DO NOT PROCEED - Torque limit would be exceeded
│ 2. Reduce user load or step height
│ 3. Consider assisted mode or mechanical support
│ 4. Run full dynamics simulation before physical testing
"""
        elif safety_status['severity'] == 'WARNING':
            report += """│ 1. ⚠ Proceed with extreme caution
│ 2. Monitor joint torques in real-time
│ 3. Have emergency stop ready
│ 4. Consider reducing step height or load
"""
        else:
            report += """│ 1. ✓ Safe to proceed with nominal operation
│ 2. Continue monitoring torque sensors
│ 3. Log telemetry for model validation
│ 4. Standard safety protocols apply
"""

        report += "━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n"
        return report


# ============================================================================
# MAIN GOVERNANCE SYSTEM
# ============================================================================

class H2ESystem:
    """Main H2E Governance System - integrates all components"""

    def __init__(self, config: Optional[H2EConfig] = None):
        self.config = config or H2EConfig()

        # First verify reproducibility with a clean seed
        print(f"\n🚀 H2E Governance System v2.0.2")
        print(f"   Session ID: {datetime.now().strftime('%Y%m%d_%H%M%S')}")
        print(f"   Seed: {self.config.SEED}")

        # Verify reproducibility BEFORE any other operations
        self.reproducible = ReproducibilityEngine.verify(self.config.SEED)

        # NOW initialize subsystems (they will manage their own RNG state)
        self.jepa = HumanoidJEPA(self.config)
        self.torque_estimator = TorqueEstimator(self.config)
        self.safety_governor = SafetyGovernor(self.config)
        self.audit = AuditSystem(self.config)
        self.diagnostic = DiagnosticEngine(self.config)

    def evaluate_mission(self, telemetry: np.ndarray, goal: str) -> Dict[str, Any]:
        """
        Evaluate mission safety and generate response

        Args:
            telemetry: 22-DoF sensor readings
            goal: Mission description string

        Returns:
            Dictionary with complete evaluation results
        """
        # Parse scenario
        scenario = Scenario.from_goal(goal)

        # Step 1: JEPA prediction
        raw_delta = self.jepa.compute_delta(telemetry)

        # Step 2: Torque estimation
        estimated_torque = self.torque_estimator.estimate_knee_torque(
            scenario.user_mass_kg,
            scenario.step_height_m,
            scenario.gait_type
        )
        safety_status = self.torque_estimator.get_safety_status(estimated_torque)

        # Step 3: Safety governor
        clamped_delta, veto = self.safety_governor.apply_staged_transition(
            raw_delta,
            safety_status['safe']
        )

        # Step 4: Diagnostic report
        report = self.diagnostic.generate_report(
            scenario, raw_delta, clamped_delta,
            veto, estimated_torque, safety_status
        )

        # Step 5: Audit logging
        self.audit.log_decision(
            scenario, raw_delta, clamped_delta,
            veto, estimated_torque, safety_status,
            claude_used=(self.diagnostic.client is not None)
        )

        # Compile results
        results = {
            'timestamp': datetime.now().isoformat(),
            'scenario': asdict(scenario),
            'metrics': {
                'raw_delta': raw_delta,
                'clamped_delta': clamped_delta,
                'estimated_torque': estimated_torque,
                'torque_limit': self.config.MAX_TORQUE_NM,
                'safety_margin': self.config.MAX_TORQUE_NM - estimated_torque,
                'veto_triggered': veto
            },
            'safety_status': safety_status,
            'report': report,
            'reproducible': self.reproducible
        }

        self.audit.record_metrics(results['metrics'])
        return results

    def shutdown(self):
        """Graceful shutdown - save metrics"""
        self.audit.save_metrics()
        print(f"\n📊 Metrics saved to {self.config.METRICS_FILE}")
        print(f"📝 Audit log: {self.config.AUDIT_FILE}")


# ============================================================================
# DEMO / TESTING
# ============================================================================

def run_demo():
    """Run complete demonstration of H2E system"""

    print("=" * 70)
    print("H2E GOVERNANCE SYSTEM - INDUSTRIAL DEMONSTRATION")
    print("=" * 70)

    # Initialize system - verification happens first, BEFORE any model init
    system = H2ESystem()

    # Test scenarios
    test_scenarios = [
        "Stair ascent (10cm)",
        "Stair descent (15cm)",
        "Level walking",
        "Stair ascent (20cm) - Heavy load"
    ]

    # Generate consistent telemetry (same for all tests)
    mock_telemetry = np.random.uniform(-1, 1, H2EConfig.SENSOR_DIM)

    # Run evaluations
    for i, goal in enumerate(test_scenarios, 1):
        print(f"\n{'='*70}")
        print(f"TEST {i}: {goal}")
        print(f"{'='*70}")

        results = system.evaluate_mission(mock_telemetry, goal)

        # Print summary
        m = results['metrics']
        s = results['safety_status']

        print(f"\n📊 QUICK SUMMARY:")
        print(f"   Torque: {m['estimated_torque']:.1f}/{m['torque_limit']} Nm "
              f"({m['safety_margin']:+.1f} Nm)")
        print(f"   Status: {s['severity']} - {s['action']}")
        print(f"   Veto: {'ACTIVE' if m['veto_triggered'] else 'inactive'}")
        print(f"   Delta: {m['clamped_delta']:.2f} (raw: {m['raw_delta']:.2f})")

        # Show report
        print(f"\n📋 DIAGNOSTIC REPORT:")
        print(results['report'])

        print("-" * 70)

    # Clean shutdown
    system.shutdown()

    print("\n" + "=" * 70)
    print("DEMONSTRATION COMPLETE")
    print("=" * 70)


if __name__ == "__main__":
    # Suppress warnings
    warnings.filterwarnings("ignore")
    logging.getLogger("anthropic").setLevel(logging.ERROR)

    # Run demo
    run_demo()

H2E GOVERNANCE SYSTEM - INDUSTRIAL DEMONSTRATION

🚀 H2E Governance System v2.0.2
   Session ID: 20260310_184651
   Seed: 123

🔍 SMLA Verification:
   Generated: [[-0.11146711558103561, 0.12036294490098953], [-0.3696345090866089, -0.2404179722070694]]
   Expected:  [[-0.11150000244379044, 0.12039999663829803], [-0.36959999799728394, -0.24040000140666962]]
   Status:    ✅ VERIFIED
✅ Claude API initialized

TEST 1: Stair ascent (10cm)

📊 QUICK SUMMARY:
   Torque: 70.0/120.0 Nm (+50.0 Nm)
   Status: NOMINAL - PROCEED
   Veto: ACTIVE
   Delta: 4.00 (raw: 8.70)

📋 DIAGNOSTIC REPORT:
Safety Assessment:

The safety audit indicates that the H2E system is operating within nominal parameters for the given task of stair ascent with a 10 cm step height. The key metrics are:

1. Clamped Delta: 4.00 (max safe: 4.0) - This is at the maximum safe limit, indicating the system is operating close to the edge of its capabilities.
2. Veto Active: True - This indicates the system has engaged its safety veto 